# 📱 Выпускной проект 2: SaaS Продуктовая аналитика и A/B-тестирование

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_15_portfolio_project_2/project_2_saas_product_analytics.ipynb)

---

## 📌 Паспорт проекта
- **Продукт**: B2B SaaS платформа аналитики (2 000 компаний, датасет `saas_users.csv`).
- **Бизнес-задача**: Оценить факторы оттока клиентов (Churn Rate) и провести статистический аудит A/B-теста новой формы подписки (`ab_test_checkout.csv`).


In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chisquare

# 1. Загрузка данных A/B теста
ab_df = pd.read_csv(r"d:\vibe_app\main_data_analytics\course\datasets\raw\ab_test_checkout.csv")

# 2. Проверка SRM (Sample Ratio Mismatch)
counts = ab_df['test_group'].value_counts()
stat_srm, p_srm = chisquare(counts.values, [len(ab_df)/2, len(ab_df)/2])
assert p_srm > 0.01, "ОШИБКА: Обнаружен SRM! Тест невалиден."
print(f" SRM-проверка пройдена успешно (p = {p_srm:.4f})")

# 3. Анализ Uplift конверсии
a = ab_df[ab_df['test_group'] == 'A_Control']
b = ab_df[ab_df['test_group'] == 'B_Variant']

cr_a = a['converted'].mean()
cr_b = b['converted'].mean()
uplift = (cr_b - cr_a) / cr_a * 100

z_stat, p_val = proportions_ztest([b['converted'].sum(), a['converted'].sum()], [len(b), len(a)])
print(f"Контроль (A): {cr_a*100:.2f}% | Тест (B): {cr_b*100:.2f}% | Uplift: +{uplift:.2f}%")
print(f"Z-stat: {z_stat:.4f}, p-value: {p_val:.5f}")

if p_val < 0.05:
    print(" РЕШЕНИЕ: Раскатывать вариант B на 100% базы.")
